In [1]:
# Cell 1 — Setup + GPU check
from pathlib import Path
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

PROJECT_ROOT = Path("/teamspace/studios/this_studio/accessops_coco_ai")
CSV_PATH = PROJECT_ROOT / "artifacts" / "captions_clean_with_splits.csv"
EDA_SUMMARY_PATH = PROJECT_ROOT / "artifacts" / "stage1b_eda" / "eda_summary.json"

OUT_DIR = PROJECT_ROOT / "artifacts" / "stage1c_preprocess"
OUT_DIR.mkdir(parents=True, exist_ok=True)

gpus = tf.config.list_physical_devices("GPU")
print("TF:", tf.__version__)
print("GPUs:", gpus)
assert len(gpus) > 0, "No GPU detected. Attach GPU before continuing."

print("CSV exists:", CSV_PATH.exists())
print("EDA summary exists:", EDA_SUMMARY_PATH.exists())


2026-04-06 17:36:54.684022: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-06 17:36:54.707290: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-06 17:36:54.737412: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-06 17:36:54.737456: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-06 17:36:54.757650: I tensorflow/core/platform/cpu_feature_gua

TF: 2.16.2
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
CSV exists: True
EDA summary exists: True


In [2]:
# Cell 2 — Load full data (no subset)
df = pd.read_csv(CSV_PATH)
df["comment_clean"] = df["comment_clean"].astype(str).str.strip().str.lower()
df["split"] = df["split"].astype(str)

df = df[df["comment_clean"].str.len() > 0].reset_index(drop=True)

print("Rows total:", len(df))
print(df["split"].value_counts())
assert {"train", "val", "test"}.issubset(set(df["split"].unique()))


Rows total: 616767
split
train    591753
val       12508
test      12506
Name: count, dtype: int64


In [3]:
# Cell 3 — No-compromise preprocessing config
# Larger vocab + longer sequence length (quality-first)
VOCAB_SIZE = 30000
MAX_LEN = 30
OOV_TOKEN = "<unk>"
PADDING = "post"
TRUNCATING = "post"

def wrap_caption(c: str) -> str:
    return f"<start> {c} <end>"

# Important for caption generation quality:
# no stopword removal, no stemming, no lemmatization
# we preserve natural language structure
df["caption_wrapped"] = df["comment_clean"].map(wrap_caption)

train_texts = df.loc[df["split"] == "train", "caption_wrapped"].tolist()
print("Train caption rows:", len(train_texts))


Train caption rows: 591753


In [5]:
# Cell 4 — Fit tokenizer on full training captions
tokenizer = Tokenizer(
    num_words=VOCAB_SIZE,
    oov_token=OOV_TOKEN,
    filters="",
    lower=True
)
tokenizer.fit_on_texts(train_texts)

start_id = tokenizer.word_index.get("<start>")
end_id = tokenizer.word_index.get("<end>")
assert start_id is not None and end_id is not None, "Missing <start>/<end> token IDs."

print("Observed vocab size:", len(tokenizer.word_index))
print("Configured vocab size:", VOCAB_SIZE)
print("start_id:", start_id, "end_id:", end_id)

Observed vocab size: 29078
Configured vocab size: 30000
start_id: 3 end_id: 4


In [6]:
# Cell 5 — Sequence conversion + quality stats
all_texts = df["caption_wrapped"].tolist()
seqs = tokenizer.texts_to_sequences(all_texts)
seq_lens_raw = np.array([len(s) for s in seqs], dtype=np.int32)

seqs_padded = pad_sequences(
    seqs, maxlen=MAX_LEN, padding=PADDING, truncating=TRUNCATING
)

truncation_rate = float((seq_lens_raw > MAX_LEN).mean())
padding_rate = float((seq_lens_raw < MAX_LEN).mean())

# Token coverage for configured vocab
word_counts = tokenizer.word_counts
total_tokens = int(sum(word_counts.values()))
kept_tokens = 0
for w, c in word_counts.items():
    idx = tokenizer.word_index.get(w, 10**9)
    if idx < VOCAB_SIZE:
        kept_tokens += c
token_coverage = float(kept_tokens / max(total_tokens, 1))

split_len_stats = (
    pd.DataFrame({"split": df["split"], "seq_len_raw": seq_lens_raw})
    .groupby("split")["seq_len_raw"]
    .agg(["count", "min", "max", "mean", "median"])
    .reset_index()
)

print("truncation_rate:", round(truncation_rate, 4))
print("padding_rate:", round(padding_rate, 4))
print("token_coverage:", round(token_coverage, 4))
display(split_len_stats)


truncation_rate: 0.0009
padding_rate: 0.999
token_coverage: 1.0


,split,count,min,max,mean,median
0,test,12506,9,51,12.417640,12.0
1,train,591753,7,51,12.454947,12.0
2,val,12508,9,36,12.457867,12.0


In [7]:
# Cell 6 — Save outputs
# top vocab preview
top_vocab_preview = (
    pd.DataFrame(tokenizer.word_counts.items(), columns=["token", "count"])
    .sort_values("count", ascending=False)
    .head(500)
)
top_vocab_preview.to_csv(OUT_DIR / "top_vocab_preview.csv", index=False)

# sequence preview
preview = df[["split", "image_name", "comment_clean", "caption_wrapped"]].copy()
preview["seq_len_raw"] = seq_lens_raw
preview["seq_preview"] = [s[:15] for s in seqs_padded.tolist()]
preview.head(200).to_csv(OUT_DIR / "sequence_preview.csv", index=False)

config = {
    "tokenization": {
        "level": "word",
        "oov_token": OOV_TOKEN,
        "use_start_end_tokens": True,
        "remove_stopwords": False,
        "stemming": False,
        "lemmatization": False
    },
    "vocab": {
        "selected_vocab_size": VOCAB_SIZE,
        "observed_vocab_size": int(len(tokenizer.word_index))
    },
    "sequence": {
        "max_length": MAX_LEN,
        "padding": PADDING,
        "truncating": TRUNCATING
    }
}

stats = {
    "rows_total": int(len(df)),
    "rows_by_split": {k: int(v) for k, v in df["split"].value_counts().to_dict().items()},
    "start_token_id": int(start_id),
    "end_token_id": int(end_id),
    "truncation_rate": truncation_rate,
    "padding_rate": padding_rate,
    "token_coverage": token_coverage
}

(OUT_DIR / "preprocess_config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
(OUT_DIR / "preprocess_stats.json").write_text(json.dumps(stats, indent=2), encoding="utf-8")
(OUT_DIR / "tokenizer.json").write_text(tokenizer.to_json(), encoding="utf-8")
split_len_stats.to_csv(OUT_DIR / "split_sequence_length_stats.csv", index=False)

print("Saved files to:", OUT_DIR)


Saved files to: /teamspace/studios/this_studio/accessops_coco_ai/artifacts/stage1c_preprocess


In [8]:
# Cell 7 — Gate check
required = [
    OUT_DIR / "preprocess_config.json",
    OUT_DIR / "preprocess_stats.json",
    OUT_DIR / "tokenizer.json",
    OUT_DIR / "top_vocab_preview.csv",
    OUT_DIR / "sequence_preview.csv",
    OUT_DIR / "split_sequence_length_stats.csv",
]
for p in required:
    assert p.exists(), f"Missing: {p}"

print("STAGE 1C PASS")


STAGE 1C PASS
